# Quality-Only IQA Training

This notebook trains **only the Quality Assessment task** without scene or distortion classification.

The model learns to predict image quality scores directly from images, without multi-task training.

**Dataset Configuration**: 
- `use_scene_labels=False`
- `use_distortion_labels=False`

## Configuration

In [ ]:
# Configuration parameters
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Training configuration
DATASET_PATHS = ["datasets/koniq-10k/"]  # Change to your dataset
OUTPUT_DIR = "outputs/10291300_quality_only"
BASE_MODEL = "src/owl3"

# Training hyperparameters
MAX_STEPS = -1  # Number of steps (-1 for full epochs)
NUM_TRAIN_EPOCHS = 3
BATCH_SIZE = 1
GRAD_ACCUM = 8
LEARNING_RATE = 2e-4
EVAL_STEPS = 100
SAVE_STEPS = 100
LOGGING_STEPS = 50

# Early stopping configuration
EARLY_STOPPING_PATIENCE = 5  # Stop if no improvement after 5 evaluations

# LoRA parameters
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

# Loss weights
USE_FIDELITY_LOSS = True

print("✅ Configuration set!")
print(f"📁 Dataset: {DATASET_PATHS}")
print(f"📁 Output: {OUTPUT_DIR}")
print(f"🎯 Training: Quality Assessment ONLY (no scene/distortion)")
print(f"🎯 Epochs: {NUM_TRAIN_EPOCHS}, max {MAX_STEPS} steps")
print(f"⚙️  Batch Size: {BATCH_SIZE} × {GRAD_ACCUM} = {BATCH_SIZE * GRAD_ACCUM}")
print(f"🛑 Early Stopping: patience={EARLY_STOPPING_PATIENCE}")

## Imports and Setup

In [ ]:
import sys
from pathlib import Path
import torch

from transformers import (
    AutoTokenizer,
    TrainingArguments,
    set_seed,
)

# Add src to path
sys.path.insert(0, str(Path.cwd()))

from src.new_train.model_wrapper import IQAModelWrapper
from src.new_train.dataset_adapter import IQAPairDataset, collate_fn_pair
from src.new_train.processor_no_cut import create_processor_no_cut
from src.new_train.iqa_trainer import IQATrainer
from src.new_train.plot_utils import plot_training_curves

# Import collate functions
from src.new_train.train_scene import collate_fn_scene
from src.new_train.train_distortion import collate_fn_distortion

# Set seed
set_seed(42)

print("✅ Imports completed!")

## Initialize Model (Run Once)

In [ ]:
# Create output directory
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

print("🔧 Loading tokenizer and processor...")
tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    trust_remote_code=True,
)
processor = create_processor_no_cut(tokenizer)

print("🔧 Initializing model with LoRA...")
model = IQAModelWrapper(
    model_name_or_path=BASE_MODEL,
    lora_r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    weight_fidelity=1.0 if USE_FIDELITY_LOSS else 0.0,
)

print("\n✅ Model initialized!")
print(f"📊 Training: Quality Assessment ONLY")

In [ ]:
print("="*80)
print("STAGE 2/3: Distortion Classification Training")
print("="*80)
print("Building on Scene classification knowledge...")

# Create dataset
print("\n📊 Creating distortion classification dataset...")
train_dataset_distortion = IQAPairDataset(
    dataset_paths=dataset_paths,
    processor=processor,
    tokenizer=tokenizer,
    split="training",
    use_scene_labels=False,
    use_distortion_labels=True,
)

val_dataset_distortion = IQAPairDataset(
    dataset_paths=dataset_paths,
    processor=processor,
    tokenizer=tokenizer,
    split="validation",
    use_scene_labels=False,
    use_distortion_labels=True,
)

print(f"✅ Training dataset size: {len(train_dataset_distortion)}")
print(f"✅ Validation dataset size: {len(val_dataset_distortion)}")

In [ ]:
# Training arguments for Distortion task
output_dir_distortion = f"{OUTPUT_DIR}/02_distortion"
training_args_distortion = TrainingArguments(
    output_dir=output_dir_distortion,
    num_train_epochs=NUM_TRAIN_EPOCHS if MAX_STEPS <= 0 else 1,
    max_steps=MAX_STEPS if MAX_STEPS > 0 else -1,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    weight_decay=0.0,
    logging_steps=LOGGING_STEPS,
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=2,
    bf16=True,
    dataloader_num_workers=12,
    remove_unused_columns=False,
    report_to="none",
    load_best_model_at_end=True,  # Load best model based on eval_loss
    metric_for_best_model="eval_loss",
    greater_is_better=False,  # Lower loss is better
)

print("✅ Training arguments configured for Distortion task")
print("   📌 Will load best model (lowest eval_loss) at end")
print(f"   📌 Early stopping: patience={EARLY_STOPPING_PATIENCE}")

In [ ]:
# Custom trainer for distortion task
class DistortionTrainer(IQATrainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        outputs = model.forward_distortion_task(
            pixel_values_A=inputs["pixel_values_A"],
            input_ids_distortion_A=inputs["input_ids_distortion_A"],
            attention_mask_distortion_A=inputs["attention_mask_distortion_A"],
            labels_distortion_A=inputs["labels_distortion_A"],
            media_offset_A=inputs["media_offset_A"],
            pixel_values_B=inputs["pixel_values_B"],
            input_ids_distortion_B=inputs["input_ids_distortion_B"],
            attention_mask_distortion_B=inputs["attention_mask_distortion_B"],
            labels_distortion_B=inputs["labels_distortion_B"],
            media_offset_B=inputs["media_offset_B"],
        )
        loss = outputs["loss"]
        return (loss, outputs) if return_outputs else loss
    
    def prediction_step(self, model, inputs, prediction_loss_only: bool, ignore_keys=None):
        has_labels = "labels_distortion_A" in inputs and "labels_distortion_B" in inputs
        with torch.no_grad():
            if has_labels:
                loss, outputs = self.compute_loss(model, inputs, return_outputs=True)
                loss = loss.mean().detach()
            else:
                loss = None
        return (loss, None, None)

print("✅ DistortionTrainer class defined!")

In [ ]:
# Create distortion trainer
early_stopping_callback = EarlyStoppingCallback(
    early_stopping_patience=EARLY_STOPPING_PATIENCE,
    early_stopping_threshold=0.0,
)

trainer_distortion = DistortionTrainer(
    model=model,
    args=training_args_distortion,
    train_dataset=train_dataset_distortion,
    eval_dataset=val_dataset_distortion,
    data_collator=collate_fn_distortion,
    callbacks=[early_stopping_callback],
)

print("✅ Distortion trainer created with early stopping!")

In [ ]:
# Train Distortion task
print("\n🚀 Starting Distortion classification training...")
print("="*80)
trainer_distortion.train()
print("="*80)
print("\n✅ Distortion training completed!")

# Generate plots
print("\n📊 Generating training curves...")
plot_training_curves(output_dir=output_dir_distortion)
print(f"✅ Plots saved to {output_dir_distortion}/")

---
## Quality Assessment Training

Direct quality score prediction without multi-task learning

In [ ]:
print("="*80)
print("Quality Assessment Training (No Multi-Task)")
print("="*80)
print("Direct quality score prediction without scene/distortion classification...")

# Create dataset
print("\n📊 Creating quality assessment dataset...")
train_dataset_quality = IQAPairDataset(
    dataset_paths=dataset_paths,
    processor=processor,
    tokenizer=tokenizer,
    split="training",
    use_scene_labels=False,
    use_distortion_labels=False,
)

val_dataset_quality = IQAPairDataset(
    dataset_paths=dataset_paths,
    processor=processor,
    tokenizer=tokenizer,
    split="validation",
    use_scene_labels=False,
    use_distortion_labels=False,
)

print(f"✅ Training dataset size: {len(train_dataset_quality)}")
print(f"✅ Validation dataset size: {len(val_dataset_quality)}")

In [ ]:
# Training arguments for Quality task
training_args_quality = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_TRAIN_EPOCHS if MAX_STEPS <= 0 else 1,
    max_steps=MAX_STEPS if MAX_STEPS > 0 else -1,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    weight_decay=0.0,
    logging_steps=LOGGING_STEPS,
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=2,
    bf16=True,
    dataloader_num_workers=12,
    remove_unused_columns=False,
    report_to="none",
    load_best_model_at_end=True,  # Load best model based on eval_plcc
    metric_for_best_model="eval_plcc",
    greater_is_better=True,  # Higher PLCC is better
)

print("✅ Training arguments configured for Quality task")
print("   📌 Will load best model (highest eval_plcc) at end")

In [ ]:
# Create trainer for quality task (uses standard IQATrainer)
early_stopping_callback = EarlyStoppingCallback(
    early_stopping_patience=EARLY_STOPPING_PATIENCE,
    early_stopping_threshold=0.0,
)

trainer_quality = IQATrainer(
    model=model,
    args=training_args_quality,
    train_dataset=train_dataset_quality,
    eval_dataset=val_dataset_quality,
    data_collator=collate_fn_pair,
    tokenizer=tokenizer,
    callbacks=[early_stopping_callback],
)

print("✅ Quality trainer created with early stopping!")

In [ ]:
# Train Quality task
print("\n🚀 Starting Quality assessment training...")
print("="*80)
trainer_quality.train()
print("="*80)
print("\n✅ Quality training completed!")

# Generate plots
print("\n📊 Generating training curves...")
try:
    from src.new_train.plot_utils import plot_metrics_summary, plot_correlation_metrics
    plot_training_curves(OUTPUT_DIR)
    plot_metrics_summary(OUTPUT_DIR)
    plot_correlation_metrics(OUTPUT_DIR)
    print("✅ All plots saved!")
except Exception as e:
    print(f"⚠️  Could not generate all plots: {e}")
    plot_training_curves(OUTPUT_DIR)

print(f"✅ Plots saved to {OUTPUT_DIR}/")

---
## Save Final Model

In [ ]:
print("="*80)
print("SAVING FINAL MODEL")
print("="*80)

final_path = f"{OUTPUT_DIR}/final_model"
model.model.save_pretrained(final_path)
tokenizer.save_pretrained(final_path)

print(f"✅ Final model saved to: {final_path}")
print("\n" + "="*80)
print("🎉 SEQUENTIAL TRAINING PIPELINE COMPLETED!")
print("="*80)
print(f"\n📊 Results:")
print(f"  Stage 1 (Scene):      {output_dir_scene}/")
print(f"  Stage 2 (Distortion): {output_dir_distortion}/")
print(f"  Stage 3 (Quality):    {output_dir_quality}/")
print(f"  Final Model:          {final_path}/")
print()

---
## Evaluation (Optional)

Evaluate the final model on test set

In [ ]:
# You can evaluate on test set here if needed
# Example:
# test_dataset = IQAPairDataset(
#     dataset_paths=dataset_paths,
#     processor=processor,
#     tokenizer=tokenizer,
#     split="testing",
# )
# 
# test_results = trainer_quality.evaluate(test_dataset)
# print(test_results)

print("💡 To evaluate the model, use the eval_sequential_model.py script:")
print(f"   python eval_sequential_model.py --model_path {final_path} --dataset_paths {' '.join(DATASET_PATHS)} --split testing")